Compute mean connectivity from HCP7T dataset -- GP-EXCLUDED tractography

Corticostriatal connectivity from GP-excluded tractography (iFOD2, 10M streamlines, SIFT2; connectivity = mu-scaled SIFT2 sum / striatal node volume) for four cortical networks:

- **Auditory**: HG, PP, PT, STGa, STGp
- **Prefrontal**: OFC, ACC, FMC, pars triangularis (IFGt), pars opercularis (IFGo), MFG, SFG, FP
- **Visual**: LOCsup, LOCinf, IntraCalc, TempOccFus, OccFus, SupraCalc, OccPole
- **Motor** (positive control): PreCG

For each network: RM-ANOVAs on striatal division (caudate vs. putamen) x striatal axis (rostral vs. caudal) x cortical ROI x hemisphere, two-way follow-ups (Greenhouse-Geisser, partial eta^2) and Wilcoxon signed-rank posthocs (FDR). Then a direct comparison across networks, auditory and visual hierarchy analyses (absolute and proportional connectivity), per-cell means that record the direction of each effect, and a log-transform sensitivity analysis.

`ATLAS_VERSION` (config cell) selects the 54-region atlas and original tractograms ('v1') or the 70-region re-tracked data ('v2', which adds ROIs to the auditory, visual and motor networks). Outputs go to stats_exports/gpexcl_visual[-v2]/ and figures/gpexcl_visual[-v2]/ so they don't overwrite the primary gpexcl results or each other.

In [ ]:
import os

import numpy as np
import pandas as pd
import seaborn as sns
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.colors as colors

from glob import glob

In [ ]:
tian_scale = 'S2'

# 'v1': 54-region atlas, original GP-excluded tractograms.
# 'v2': 70-region atlas (adds cuneal, lingual, temporo-occipital MTG/ITG,
#       posterior MTG, temporal pole, postcentral gyrus, SMA), re-tracked with
#       run_subject_gpexcl_v2.sh. Every atlas / tractogram / output path below
#       follows from this switch, so v1 and v2 results never overwrite each other.
ATLAS_VERSION = 'v2'

_v = '' if ATLAS_VERSION == 'v1' else f'-{ATLAS_VERSION}'
ATLAS_BASE = f'atlas-custom_subcort-tian{tian_scale}_cort-aud-vis-prefrontal{_v}'  # MNI atlas + ROI masks
ATLAS_DESC = f'atlas-custom_subcort-tian{tian_scale}_cort-carpet{_v}'             # subject-space atlas / connectome desc
SL_BASE    = f'streamlines_alg-iFOD2_nsl-10mil_gpexcl{_v}'                        # tractogram / SIFT2 / connectome tag
OUT_TAG    = f'gpexcl_visual{_v}'                                                 # stats_exports/ and figures/ subdir

lut_fpath = os.path.join('/Users/dsj3886/data_local/derivatives/', ATLAS_BASE, f'{ATLAS_BASE}_lut.tsv')

region_list = list(pd.read_csv(lut_fpath, sep='\t', header=None).iloc[:,1])

In [ ]:
# Data preparation (all cortical ROIs to keep)
AUDITORY_ROIS   = ['STGa', 'PP', 'HG', 'PT', 'STGp']
PREFRONTAL_ROIS = ['OFC', 'ACC', 'FMC', 'IFGt', 'IFGo', 'MFG', 'SFG', 'FP']
VISUAL_ROIS     = ['LOCsup', 'LOCinf', 'IntraCalc', 'TempOccFus', 'OccFus', 'SupraCalc', 'OccPole']
MOTOR_ROIS      = ['PreCG']
if ATLAS_VERSION == 'v2':
    AUDITORY_ROIS = AUDITORY_ROIS + ['MTGp', 'TempPole']
    VISUAL_ROIS   = VISUAL_ROIS + ['Cuneal', 'Lingual', 'MTGto', 'ITGto']
    MOTOR_ROIS    = MOTOR_ROIS + ['PostCG', 'SMA']
CORTEX_ROIS     = AUDITORY_ROIS + PREFRONTAL_ROIS + VISUAL_ROIS + MOTOR_ROIS

NETWORKS = ['Auditory', 'Prefrontal', 'Visual', 'Motor']

# Display-only relabeling: the codes above stay unchanged (they must match the
# external atlas LUT / connectivity data); this is applied only when rendering
# text (plot titles, tick labels, annotations).
ROI_DISPLAY_NAMES = {'IFGo': 'ParsOp', 'IFGt': 'ParsTri'}

def _display_label(name):
    if len(name) > 2 and name[:2] in ('L-', 'R-'):
        return name[:2] + ROI_DISPLAY_NAMES.get(name[2:], name[2:])
    return ROI_DISPLAY_NAMES.get(name, name)

In [ ]:
data_dir = os.path.join('/Users/dsj3886/data_local/derivatives/',
                        'HCP_7T_diffusion/msmt_csd_nthreads-1/')


STATS_OUT_DIR = os.path.join('stats_exports', OUT_TAG)
os.makedirs(STATS_OUT_DIR, exist_ok=True)

FIGURES_DIR = os.path.join('figures', OUT_TAG)
os.makedirs(FIGURES_DIR, exist_ok=True)

## Read in participant connectivity matrices

In [ ]:
print(data_dir + f'/*/connectome_{SL_BASE}/{ATLAS_DESC}_sift2-noscaling/{SL_BASE}_connectome_sift2-noscaling.csv')

In [ ]:
conn_files = sorted(glob(data_dir + f'/*/connectome_{SL_BASE}/{ATLAS_DESC}_sift2-noscaling/'
                         f'{SL_BASE}_connectome_sift2-noscaling.csv'))

# derive participant IDs directly from conn_files (preserves same order)
participants_with_files = sorted(list({os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(p)))) for p in conn_files}))

In [ ]:
print(len(conn_files))

In [ ]:
test_df = pd.read_csv(conn_files[0], header=None)
print(test_df)
print(len(test_df))


In [ ]:
num_regions = len(region_list)  # 54
conn_files_54 = [f for f in conn_files
                 if pd.read_csv(f, header=None).shape[0] == num_regions]

num_subjects = len(conn_files_54)
connmats_raw = np.zeros((num_subjects, num_regions, num_regions))
slcounts = np.zeros((num_regions, num_subjects))
list_df = []

for sx, matrix_fpath in enumerate(conn_files_54):
    sub_df = pd.read_csv(matrix_fpath, header=None)
    list_df.append(sub_df)
    connmats_raw[sx,:,:] = sub_df.to_numpy()
    slcounts[:,sx] = sub_df.to_numpy().sum(axis=0)

In [ ]:
# --- Per-subject connectome normalisation ----------------------------------
# The connectome CSVs are RAW SIFT2-weighted streamline sums (tck2connectome
# run WITHOUT -scale_invnodevol). Each edge is normalised by:
#   (1) the subject's SIFT2 proportionality coefficient mu -> cross-subject scale
#   (2) the volume of the STRIATAL node only               -> connectivity density
#       C(striatum_i, cortex_j) = mu_s * raw_sum(i, j) / V_striatum(s, i)
# Striatal volumes come from each subject's own T1w/acpc-space parcellation
# (the same image passed to tck2connectome).

striatum_idx = np.array([i for i, r in enumerate(region_list)
                         if r.startswith(('aPUT', 'pPUT', 'aCAU', 'pCAU'))])

_subatlas_tmpl = ('/Users/dsj3886/data_local/derivatives/HCP_7T_diffusion/'
                  'atlas_space-sub/{sub}/sub-{sub}_'
                  f'{ATLAS_DESC}'
                  '_atlas_warp-standard2acpc_dc_space-T1w.nii.gz')
_mu_tmpl = ('/Users/dsj3886/data_local/derivatives/HCP_7T_diffusion/'
            'msmt_csd_nthreads-1/{sub}/'
            f'sift2_mu_{SL_BASE}.txt')

# subject IDs in the same order as connmats_raw (== conn_files_54)
subs_54 = [os.path.basename(os.path.dirname(os.path.dirname(os.path.dirname(f))))
           for f in conn_files_54]

mu = np.zeros(num_subjects)
striatal_vol_mm3 = np.zeros((num_subjects, num_regions))   # 0 outside striatum
for sx, sub in enumerate(subs_54):
    mu[sx] = float(np.loadtxt(_mu_tmpl.format(sub=sub)))
    seg_img = nib.load(_subatlas_tmpl.format(sub=sub))
    seg = np.asarray(seg_img.dataobj)
    vox_mm3 = float(np.prod(seg_img.header.get_zooms()[:3]))
    for idx in striatum_idx:
        striatal_vol_mm3[sx, idx] = np.count_nonzero(seg == idx + 1) * vox_mm3

print(f'mu: mean={mu.mean():.4g}  range=[{mu.min():.4g}, {mu.max():.4g}]')
print('striatal volume (mm^3), across-subject mean (SD):')
for idx in striatum_idx:
    v = striatal_vol_mm3[:, idx]
    print(f'  {region_list[idx]:9s} {v.mean():8.1f} ({v.std():6.1f})')

_sv = striatal_vol_mm3[:, striatum_idx]
if (_sv == 0).any():
    print(f'WARNING: {(_sv == 0).sum()} zero striatal volume(s) -> those edges = nan')

# scale the whole matrix by mu; divide the striatal ROWS by their own volume
inv_vol = np.ones((num_subjects, num_regions))
with np.errstate(divide='ignore'):
    inv_vol[:, striatum_idx] = np.where(_sv > 0, 1.0 / _sv, np.nan)

#connmats_corrected = connmats_raw * mu[:, None, None] * inv_vol[:, :, None]
# add a normalizing constant to get values (already a.u.) into less small decimals
connmats_corrected = 1e4 * connmats_raw * mu[:, None, None] * inv_vol[:, :, None]

In [ ]:
# Drop subjects with a zero-volume striatal ROI. A zero volume makes that
# subject's connectivity NaN for that ROI (cell above); AnovaRM aggregates
# cells with np.mean, which does NOT skip NaN, so a single bad cell corrupts
# that subject's entire design row (LinAlgError: SVD did not converge).
_bad = np.where((_sv == 0).any(axis=1))[0]
if len(_bad):
    _bad_subs = [subs_54[i] for i in _bad]
    print(f'Dropping {len(_bad)} subject(s) with a zero-volume striatal ROI: {_bad_subs}')
    _keep = np.array([i for i in range(num_subjects) if i not in set(_bad)])
    connmats_corrected = connmats_corrected[_keep]
    subs_54 = [subs_54[i] for i in _keep]
    mu = mu[_keep]
    striatal_vol_mm3 = striatal_vol_mm3[_keep]
    num_subjects = len(subs_54)

connmats = connmats_corrected


In [ ]:
np.shape(connmats)

In [ ]:
# check the streamline count matrix (region x subject)
slcounts

In [ ]:
slcounts[0,:]

## Compute mean streamline count per ROI

In [ ]:
# compute the mean streamline count per IC subdivision
mean_slcount = slcounts.mean(axis=1)
print(mean_slcount)
print(mean_slcount.shape)

In [ ]:
mean_connmat = connmats.mean(axis=0)

print(mean_connmat)
print(mean_connmat.shape)

In [ ]:

# add ROI names to df
mean_df = pd.DataFrame(mean_connmat, columns=region_list)
mean_df.insert(0, 'seed', region_list)
mean_df.insert(0, 'mean_slcount', mean_slcount)


In [ ]:
mean_df

In [ ]:
# save the mean data
out_dir = os.path.join(data_dir, f'group_plots_prefrontal_gpexcl{_v}')
out_fpath = os.path.join(out_dir, 'pd_mean_connectivity_scale-mu-strivol.txt')
os.makedirs(out_dir, exist_ok=True)
mean_df.to_csv(out_fpath, sep='\t', index=False)

#### Value distribution

No low-value threshold is applied. See the *Plot distributions* section for the distribution of the striatum×cortex edges that enter the statistics.

### Plot distributions

In [ ]:
connmats.shape

In [ ]:
# Distribution of the striatum x cortex connectivity values that enter the stats
_ctx = [i for i, r in enumerate(region_list) if r.startswith(('L-', 'R-'))]
block = connmats[:, striatum_idx][:, :, _ctx].ravel()   # subjects x striatum x cortex
block_nz = block[np.isfinite(block) & (block > 0)]

print(f'{block.size} striatum x cortex edges  |  '
      f'{np.mean(block == 0) * 100:.1f}% exactly zero  |  '
      f'{np.mean(~np.isfinite(block)) * 100:.1f}% nan')
print('\nall edges:')
print(pd.Series(block).describe(percentiles=[.05, .25, .5, .75, .95, .99]))
print('\nnon-zero edges:')
print(pd.Series(block_nz).describe(percentiles=[.05, .25, .5, .75, .95, .99]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(block_nz, bins=60, ax=axes[0])
axes[0].set(title='connectivity, non-zero edges (linear)',
            xlabel='connectivity (a.u. / mm³)')
sns.histplot(np.log10(block_nz), bins=60, ax=axes[1])
axes[1].set(title='connectivity, non-zero edges (log10)', xlabel='log10 connectivity')
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'distribution_connectivity_striatum-cortex.svg'))


## Connectivity plots

In [ ]:
num_participants = len(connmats)
print(f'Participants entering analysis: N = {num_participants}')

# Account for subjects not in the analysis. run_subject.sh skips any subject
# whose subject-space atlas has < 54 regions (warped before the prefrontal ROIs
# were added), so those have no _sift2-noscaling connectome. Check the atlas
# files directly for a complete picture.
_atlas_dir = '/Users/dsj3886/data_local/derivatives/HCP_7T_diffusion/atlas_space-sub'
_atlas_tail = f'_{ATLAS_DESC}_atlas_warp-standard2acpc_dc_space-T1w.nii.gz'
_atlas_files = sorted(glob(os.path.join(_atlas_dir, '*', f'sub-*{_atlas_tail}')))
_by_regions = {}
for _a in _atlas_files:
    _sub = os.path.basename(_a).split('_')[0].replace('sub-', '')
    _n = int(np.asarray(nib.load(_a).dataobj).max())
    _by_regions.setdefault(_n, []).append(_sub)

print(f'\nsubject-space atlases found: {len(_atlas_files)}')
for _n, _subs in sorted(_by_regions.items()):
    _flag = 'analysed' if _n == num_regions else f'EXCLUDED (re-warp with the {num_regions}-region atlas)'
    print(f'  {_n:>2}-region: {len(_subs):>3} subjects   <- {_flag}')
_excluded = sorted(s for n, ss in _by_regions.items() if n != num_regions for s in ss)
if _excluded:
    print(f'\nexcluded: {" ".join(_excluded)}')

In [ ]:
plot_df = mean_df.iloc[:, 2:]


In [ ]:
plot_df

In [ ]:
roi_dir = os.path.join('/Users/dsj3886/data_local/derivatives/', ATLAS_BASE)

In [ ]:
print(roi_dir)

In [ ]:
cau_lh_list = sorted(glob(roi_dir+'/*CAU*lh.nii.gz'))
put_lh_list = sorted(glob(roi_dir+'/*PUT*lh.nii.gz'))
cau_rh_list = sorted(glob(roi_dir+'/*CAU*rh.nii.gz'))
put_rh_list = sorted(glob(roi_dir+'/*PUT*rh.nii.gz'))

cp_list = cau_lh_list + put_lh_list + cau_rh_list + put_rh_list

In [ ]:
cp_list

In [ ]:
cp_roi_list = [os.path.basename(x).split('.')[0].split(ATLAS_BASE + '_')[1] for x in cp_list]

print(cp_roi_list)

In [ ]:
cp_sl_list = []
for rx, roi_name in enumerate(cp_roi_list):
    print(roi_name)
    roi_row = mean_df[mean_df.seed==roi_name]
    print(roi_row)

    roi_row_corrected = roi_row.iloc[0,2:].to_numpy() 
    print(roi_row_corrected)

    cp_sl_list.append(roi_row_corrected)

In [ ]:
cs_df = pd.DataFrame(data=cp_sl_list, 
                          columns=region_list,
                          index=cp_roi_list)
print(cs_df)

In [ ]:
region_list

In [ ]:
df_max = cs_df.to_numpy().max()

sns.set(font_scale=.7)
sns.set_style("white")

f, ax = plt.subplots(1,1,figsize=(6,3), dpi=300)

sns.heatmap(cs_df, ax=ax, cmap="afmhot_r",
            norm=colors.LogNorm(vmin=np.nanmin(cs_df.values[cs_df.values > 0]), vmax=df_max),
            square=True)
ax.set_title(f'Auditory–striatal probabilistic connectivity\n(mean of {num_participants} HCP 7T participants)');
#ax.tick_params(rotation=45, ha='right');
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

#sns.heatmap(right_cs_df, ax=ax[1], cmap="afmhot_r",
#            norm=colors.LogNorm(vmin=np.nanmin(cs_df.values[cs_df.values > 0]), vmax=df_max),)
#ax[1].tick_params(rotation=45);
#ax[1].set_title('Right hemisphere auditory–striatal probabilistic connectivity');
f.tight_layout()

In [ ]:
cortical_cols = [r for r in cs_df.columns if r.startswith(('L-', 'R-'))]

# Prefrontal ROIs were added with alternating L/R pairs; regroup to L-all then R-all
_pfc_base = {'FP', 'SFG', 'MFG', 'IFGt', 'IFGo', 'PreCG', 'FMC', 'PCG', 'ACC', 'OFC', 'FO'}
pfc_cols  = [c for c in cortical_cols if c[2:] in _pfc_base]
pfc_start = cortical_cols.index(pfc_cols[0])
pfc_end   = cortical_cols.index(pfc_cols[-1]) + 1

reordered = (cortical_cols[:pfc_start]
             + [c for c in pfc_cols if c.startswith('L-')]
             + [c for c in pfc_cols if c.startswith('R-')]
             + cortical_cols[pfc_end:])

cs_df_cortical = cs_df[reordered]
cs_df_cortical_display = cs_df_cortical.copy()
cs_df_cortical_display.columns = [_display_label(c) for c in cs_df_cortical_display.columns]

df_max = cs_df_cortical.to_numpy().max()

sns.set(font_scale=.7)
sns.set_style("white")

f, ax = plt.subplots(1, 1, figsize=(12, 3), dpi=300)

sns.heatmap(cs_df_cortical_display, ax=ax, cmap="afmhot_r",
            cbar_kws={'label': 'connectivity (a.u. / mm³)', 'shrink': 0.5},
            norm=colors.LogNorm(vmin=1, vmax=df_max),
            square=True)
ax.set_title(f'Corticostriatal probabilistic connectivity\n(mean of {num_participants} HCP 7T participants)')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

f.tight_layout()
f.savefig(os.path.join(FIGURES_DIR, 'heatmap_mean-streamlines_all-regions.svg'))

In [ ]:
_aud_base = set(AUDITORY_ROIS)
no_vis_cols = [c for c in reordered if c[2:] in (_aud_base | _pfc_base)]
cs_df_no_vis = cs_df[no_vis_cols]
cs_df_no_vis_display = cs_df_no_vis.copy()
cs_df_no_vis_display.columns = [_display_label(c) for c in cs_df_no_vis_display.columns]

df_max_nov = cs_df_no_vis.to_numpy().max()

f2, ax2 = plt.subplots(1, 1, figsize=(9, 3), dpi=300)

sns.heatmap(cs_df_no_vis_display, ax=ax2, cmap="afmhot_r",
            cbar_kws={'label': 'connectivity (a.u. / mm³)', 'shrink': 0.5},
            norm=colors.LogNorm(vmin=np.nanmin(cs_df_no_vis.values[cs_df_no_vis.values > 0]), vmax=df_max_nov),
            square=True)
ax2.set_title(f'Corticostriatal probabilistic connectivity — auditory & prefrontal (mean of {num_participants} HCP 7T participants)')
plt.setp(ax2.get_xticklabels(), rotation=45, ha="right")

f2.tight_layout()
f2.savefig(os.path.join(FIGURES_DIR, 'heatmap_mean-streamlines_auditory-prefrontal-regions.svg'))

In [ ]:
df_max = cs_df.to_numpy().max()
_aud_cols = [c for c in cs_df.columns if c[:2] in ('L-', 'R-') and c[2:] in AUDITORY_ROIS]
cs_df_aud_display = cs_df[_aud_cols].copy()
cs_df_aud_display.columns = [_display_label(c) for c in cs_df_aud_display.columns]

sns.set(font_scale=.7)
sns.set_style("white")

f, ax = plt.subplots(1,1,figsize=(6,3), dpi=300)

sns.heatmap(cs_df_aud_display, ax=ax, cmap="afmhot_r",
            norm=colors.LogNorm(vmin=np.nanmin(cs_df[_aud_cols].values[cs_df[_aud_cols].values > 0]), vmax=df_max),
            cbar_kws={'label': 'connectivity (a.u. / mm³)', 'shrink': 0.5},
            square=True)
ax.set_title(f'Auditory–striatal probabilistic connectivity\n(mean of {num_participants} HCP 7T participants)');
#ax.tick_params(rotation=45, ha='right');
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

#sns.heatmap(right_cs_df, ax=ax[1], cmap="afmhot_r",
#            norm=colors.LogNorm(vmin=np.nanmin(cs_df[_aud_cols].values[cs_df[_aud_cols].values > 0]), vmax=df_max),)
#ax[1].tick_params(rotation=45);
#ax[1].set_title('Right hemisphere auditory–striatal probabilistic connectivity');

f.tight_layout()
f.savefig(os.path.join(FIGURES_DIR, 'heatmap_mean-streamlines_auditory-regions.svg'))

## Combine separate participants into single dataframe

In [ ]:
participant_df_list = []
for i in range(connmats.shape[0]):
    df = pd.DataFrame(connmats[i], index=region_list, columns=region_list)
    df['participant_id'] = i
    participant_df_list.append(df)

In [ ]:
striatum_rois = [r for r in region_list if r.startswith(('aPUT', 'pPUT', 'aCAU', 'pCAU'))]
cortex_rois = [r for r in region_list if r.startswith(('L-', 'R-'))]

filtered_df_list = []
for df in participant_df_list:
    sub_df = df.loc[striatum_rois, cortex_rois]
    sub_df['participant_id'] = df['participant_id'].iloc[0]
    filtered_df_list.append(sub_df)

In [ ]:
rostral_caudal_map = {
    'aPUT-lh': 'Rostral', 'aCAU-lh': 'Rostral',
    'pPUT-lh': 'Caudal', 'pCAU-lh': 'Caudal',
    'aPUT-rh': 'Rostral', 'aCAU-rh': 'Rostral',
    'pPUT-rh': 'Caudal', 'pCAU-rh': 'Caudal'
}

for df in filtered_df_list:
    df['Rostral_Caudal'] = df.index.map(rostral_caudal_map)

In [ ]:
caudate_putamen_map = {
    'aPUT-lh': 'Putamen', 'aCAU-lh': 'Caudate',
    'pPUT-lh': 'Putamen', 'pCAU-lh': 'Caudate',
    'aPUT-rh': 'Putamen', 'aCAU-rh': 'Caudate',
    'pPUT-rh': 'Putamen', 'pCAU-rh': 'Caudate'
}

for df in filtered_df_list:
    df['Caudate_Putamen'] = df.index.map(caudate_putamen_map)

In [ ]:
all_df = pd.concat(filtered_df_list)
all_df = all_df.reset_index().rename(columns={'index': 'Striatum_ROI'})

In [ ]:

# Melt to long format
all_long_df = all_df.melt(id_vars=['participant_id', 'Striatum_ROI', 'Rostral_Caudal', 'Caudate_Putamen'], 
                      value_vars=cortex_rois, 
                      var_name='Cortex_ROI', value_name='Connectivity')


In [ ]:
all_long_df.head()

In [ ]:
# Add hemisphere info before stripping labels
def get_striatum_hemisphere(roi):
    if roi.endswith('-lh'):
        return 'Left'
    elif roi.endswith('-rh'):
        return 'Right'
    else:
        return None

def get_cortex_hemisphere(roi):
    if roi.startswith('L-'):
        return 'Left'
    elif roi.startswith('R-'):
        return 'Right'
    else:
        return None

all_long_df['striatum_hemisphere'] = all_long_df['Striatum_ROI'].apply(get_striatum_hemisphere)
all_long_df['cortex_hemisphere'] = all_long_df['Cortex_ROI'].apply(get_cortex_hemisphere)

# Keep only rows where hemispheres match
all_long_df = all_long_df[all_long_df['striatum_hemisphere'] == all_long_df['cortex_hemisphere']]

# Optionally, create a single 'hemisphere' column and strip hemisphere labels
all_long_df['hemisphere'] = all_long_df['striatum_hemisphere']
all_long_df['Striatum_ROI'] = all_long_df['Striatum_ROI'].apply(lambda x: x[:-3] if x.endswith(('-lh', '-rh')) else x)
all_long_df['Cortex_ROI'] = all_long_df['Cortex_ROI'].apply(lambda x: x[2:] if x.startswith(('L-', 'R-')) else x)

In [ ]:
all_long_df.shape

In [ ]:
all_long_df.head()

In [ ]:
all_long_df.Cortex_ROI.unique()

In [ ]:
auditory_rois = ['HG', 'PP', 'PT', 'STGp', 'STGa']
aud_long_df = all_long_df[all_long_df['Cortex_ROI'].isin(auditory_rois)]


In [ ]:
df_collapsed = aud_long_df.groupby(["participant_id", 'Cortex_ROI', 'Striatum_ROI',], 
                                       as_index=False).agg({"Connectivity": "mean"})

fig, ax = plt.subplots(1, 1, figsize=(6,4), dpi=500)

sns.barplot(data=df_collapsed, 
              x='Cortex_ROI', y='Connectivity', 
              order=auditory_rois,
              hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
              palette='crest',  
              #fliersize=0, whis=(8, 92),
              #linewidth=1.5,
              alpha=0.8, gap=0.1, fill=False,
              #legend=None,
              ax=ax)
sns.stripplot(data=df_collapsed, 
              x='Cortex_ROI', y='Connectivity', 
              order=auditory_rois,
              hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
              palette='crest',
              dodge=True,
              jitter=True,
              linewidth=0.2,
              alpha=0.2,
              legend=None,
              ax=ax)

#ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='center')
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
ax.set_title(f'Corticostriatal connectivity by HCP participant');
plt.yscale('log')
ax.set_ylabel('connectivity (a.u. / mm³)')
#ax.set_ylim([-5, 5])

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'streamline-count_x-cortex_hue-striatum.svg'))

## Across-participant statistics

**Omnibus ANOVAs** are repeated-measures ANOVAs on the untransformed connectivity
values: `statsmodels` `AnovaRM` for the 3- and 4-factor designs, and `pingouin`
`rm_anova` (Greenhouse–Geisser correction + partial η²) for the 2-factor designs.

**Pairwise post-hoc comparisons** use the **Wilcoxon signed-rank test**
(`pg.pairwise_tests(..., parametric=False)`, Benjamini–Hochberg FDR). The
connectivity values are strongly right-skewed with a spike at zero (region pairs
with no streamlines), which breaks the normality assumption of the paired
*t*-test; the rank-based Wilcoxon test is robust to it. The omnibus ANOVAs have
**not** yet been adapted (log-transform or non-parametric) — a known limitation.


In [ ]:
from statsmodels.stats.anova import AnovaRM
import pingouin as pg

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
from stats_fmt import export_anova, export_posthoc, export_cell_means

STATS_OUT_DIR = os.path.join('stats_exports', OUT_TAG)
os.makedirs(STATS_OUT_DIR, exist_ok=True)

FIGURES_DIR = os.path.join('figures', OUT_TAG)
os.makedirs(FIGURES_DIR, exist_ok=True)

### All factors

In [ ]:
# Mixed-Effects ANOVA with all factors and interactions
aov = AnovaRM(aud_long_df, 
              aggregate_func='mean',
              depvar='Connectivity', 
              subject='participant_id', 
              within=['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI', 'hemisphere',]).fit()

# Display results
aov.summary()
export_anova(aov, 'aud_full', STATS_OUT_DIR)


### No hemisphere

In [ ]:
# Mixed-Effects ANOVA with all factors and interactions
aov = AnovaRM(aud_long_df, 
              aggregate_func='mean',
              depvar='Connectivity', 
              subject='participant_id', 
              within=['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI']).fit()

# Display results
aov.summary()
export_anova(aov, 'aud_cau_rostral_roi', STATS_OUT_DIR)


### Caudate–Putamen

In [ ]:
# Mixed-Effects ANOVA with Hemisphere, Region, and Interaction
aov = AnovaRM(aud_long_df, 
              aggregate_func='mean',
              depvar='Connectivity', 
              subject='participant_id', 
              within=['Caudate_Putamen', 'Cortex_ROI', 'hemisphere',]).fit()

# Display results
aov.summary()
export_anova(aov, 'aud_cau_roi_hemi', STATS_OUT_DIR)


#### 2-way: Cortex ROI x Caudate/Putamen

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = aud_long_df.groupby(['participant_id', 'Cortex_ROI', 'Caudate_Putamen'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Cortex_ROI', 'Caudate_Putamen'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'aud_roi_cau', STATS_OUT_DIR)


In [ ]:

# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=aud_long_df,
                             dv='Connectivity', 
                             within=[ 'Caudate_Putamen', 'Cortex_ROI'],
                             subject='participant_id',
                             padjust='fdr', parametric=False) 

# Check available columns
print("Available columns:", pairwise.columns)

# Display key columns 
print("\nPost-hoc Pairwise Comparisons:")

pd.set_option('display.max_rows', None)

#pairwise_interaction = pairwise[pairwise.Contrast.str.contains(' * ')].reset_index()
#pairwise_interaction = pairwise_interaction[pairwise_interaction['p-unc'] < 0.05].reset_index()

# show results
pairwise[['Contrast',  'Caudate_Putamen', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
#print(pairwise_interaction)
export_posthoc(pairwise, 'aud_cau_roi', STATS_OUT_DIR)


In [ ]:

# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=aud_long_df,
                             dv='Connectivity', 
                             within=['Cortex_ROI', 'Caudate_Putamen', ],
                             subject='participant_id',
                             padjust='fdr', parametric=False) 

# Check available columns
print("Available columns:", pairwise.columns)

# Display key columns 
print("\nPost-hoc Pairwise Comparisons:")

pd.set_option('display.max_rows', None)

#pairwise_interaction = pairwise[pairwise.Contrast.str.contains(' * ')].reset_index()
#pairwise_interaction = pairwise_interaction[pairwise_interaction['p-unc'] < 0.05].reset_index()

# show results
pairwise[['Contrast',  'Cortex_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
#print(pairwise_interaction)
export_posthoc(pairwise, 'aud_roi_cau', STATS_OUT_DIR)


In [ ]:

# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=aud_long_df,
                             dv='Connectivity', 
                             within=[ 'hemisphere', 'Cortex_ROI',],
                             subject='participant_id',
                             padjust='fdr', parametric=False) 

# Check available columns
print("Available columns:", pairwise.columns)

# Display key columns 
print("\nPost-hoc Pairwise Comparisons:")

pd.set_option('display.max_rows', None)

#pairwise_interaction = pairwise[pairwise_by_learning_stage.Contrast.str.contains(' * ')].reset_index()
#pairwise_interaction = pairwise_interaction[pairwise_interaction['p-unc'] < 0.05].reset_index()

# show results
pairwise[['Contrast',  'hemisphere', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
#print(pairwise_interaction)
export_posthoc(pairwise, 'aud_hemi_roi', STATS_OUT_DIR)


In [ ]:

# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=aud_long_df,
                             dv='Connectivity', 
                             within=[ 'Cortex_ROI', 'hemisphere',],
                             subject='participant_id',
                             padjust='fdr', parametric=False) 

# Check available columns
print("Available columns:", pairwise.columns)

# Display key columns 
print("\nPost-hoc Pairwise Comparisons:")

pd.set_option('display.max_rows', None)

#pairwise_interaction = pairwise[pairwise_by_learning_stage.Contrast.str.contains(' * ')].reset_index()
#pairwise_interaction = pairwise_interaction[pairwise_interaction['p-unc'] < 0.05].reset_index()

# show results
pairwise[['Contrast',  'Cortex_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
#print(pairwise_interaction)
export_posthoc(pairwise, 'aud_roi_hemi', STATS_OUT_DIR)


In [ ]:

# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=aud_long_df,
                             dv='Connectivity', 
                             within=['Striatum_ROI', 'hemisphere', ],
                             subject='participant_id',
                             padjust='fdr', parametric=False) 

# Check available columns
print("Available columns:", pairwise.columns)

# Display key columns 
print("\nPost-hoc Pairwise Comparisons:")

pd.set_option('display.max_rows', None)

#pairwise_interaction = pairwise[pairwise_by_learning_stage.Contrast.str.contains(' * ')].reset_index()
#pairwise_interaction = pairwise_interaction[pairwise_interaction['p-unc'] < 0.05].reset_index()

# show results
pairwise[['Contrast', 'Striatum_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
#print(pairwise)
export_posthoc(pairwise, 'aud_striatum_hemi', STATS_OUT_DIR)


### Rostral–caudal

In [ ]:
# Mixed-Effects ANOVA with Hemisphere, Region, and Interaction
aov = AnovaRM(aud_long_df, 
              aggregate_func='mean',
              depvar='Connectivity', 
              subject='participant_id', 
              within=['Rostral_Caudal', 'hemisphere', 'Cortex_ROI']).fit()

# Display results
aov.summary()
export_anova(aov, 'aud_rostral_hemi_roi', STATS_OUT_DIR)


#### 2-way: Cortex ROI x Rostral/Caudal

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = aud_long_df.groupby(['participant_id', 'Cortex_ROI', 'Rostral_Caudal'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Cortex_ROI', 'Rostral_Caudal'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'aud_roi_rostral', STATS_OUT_DIR)


In [ ]:

# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=aud_long_df,
                             dv='Connectivity', 
                             within=[ 'hemisphere', 'Rostral_Caudal',],
                             subject='participant_id',
                             padjust='fdr', parametric=False) 

# Check available columns
print("Available columns:", pairwise.columns)

# Display key columns 
print("\nPost-hoc Pairwise Comparisons:")

pd.set_option('display.max_rows', None)

#pairwise_interaction = pairwise[pairwise_by_learning_stage.Contrast.str.contains(' * ')].reset_index()
#pairwise_interaction = pairwise_interaction[pairwise_interaction['p-unc'] < 0.05].reset_index()

# show results
pairwise[['Contrast',  'hemisphere', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
#print(pairwise_interaction)
export_posthoc(pairwise, 'aud_hemi_rostral', STATS_OUT_DIR)


In [ ]:

# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=aud_long_df,
                             dv='Connectivity', 
                             within=['Rostral_Caudal', 'hemisphere', ],
                             subject='participant_id',
                             padjust='fdr', parametric=False) 

# Check available columns
print("Available columns:", pairwise.columns)

# Display key columns 
print("\nPost-hoc Pairwise Comparisons:")

pd.set_option('display.max_rows', None)

#pairwise_interaction = pairwise[pairwise_by_learning_stage.Contrast.str.contains(' * ')].reset_index()
#pairwise_interaction = pairwise_interaction[pairwise_interaction['p-unc'] < 0.05].reset_index()

# show results
pairwise[['Contrast', 'Rostral_Caudal', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
#print(pairwise_interaction)
export_posthoc(pairwise, 'aud_rostral_hemi', STATS_OUT_DIR)


In [ ]:

# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=aud_long_df,
                             dv='Connectivity', 
                             within=[ 'Cortex_ROI', 'Rostral_Caudal',],
                             subject='participant_id',
                             padjust='fdr', parametric=False) 

# Check available columns
print("Available columns:", pairwise.columns)

# Display key columns 
print("\nPost-hoc Pairwise Comparisons:")

pd.set_option('display.max_rows', None)

#pairwise_interaction = pairwise[pairwise_by_learning_stage.Contrast.str.contains(' * ')].reset_index()
#pairwise_interaction = pairwise_interaction[pairwise_interaction['p-unc'] < 0.05].reset_index()

# show results
pairwise[['Contrast',  'Cortex_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
#print(pairwise_interaction)
export_posthoc(pairwise, 'aud_roi_rostral', STATS_OUT_DIR)


### Auditory vs. Prefrontal vs. Visual

In [ ]:
net_df = all_long_df[all_long_df['Cortex_ROI'].isin(CORTEX_ROIS)].copy()

category_map = {r: 'Auditory' for r in AUDITORY_ROIS}
category_map.update({r: 'Prefrontal' for r in PREFRONTAL_ROIS})
category_map.update({r: 'Visual' for r in VISUAL_ROIS})
category_map.update({r: 'Motor' for r in MOTOR_ROIS})
net_df['Cortex_Category'] = net_df['Cortex_ROI'].map(category_map)

In [ ]:
net_df.groupby(['Cortex_Category', 'Caudate_Putamen'])['Connectivity'].agg(['mean', 'median'])

#### All factors

In [ ]:
aov = AnovaRM(net_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_Category', 'hemisphere']).fit()
aov.summary()
export_anova(aov, 'net_full', STATS_OUT_DIR)


#### No hemisphere

In [ ]:
aov = AnovaRM(net_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_Category']).fit()
aov.summary()
export_anova(aov, 'net_cau_rostral_cat', STATS_OUT_DIR)


#### Caudate–Putamen

In [ ]:
aov = AnovaRM(net_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Caudate_Putamen', 'Cortex_Category', 'hemisphere']).fit()
aov.summary()
export_anova(aov, 'net_cau_cat_hemi', STATS_OUT_DIR)


##### 2-way: Category x Caudate/Putamen

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = net_df.groupby(['participant_id', 'Cortex_Category', 'Caudate_Putamen'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Cortex_Category', 'Caudate_Putamen'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'net_cat_cau', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=net_df,
                             dv='Connectivity',
                             within=['Caudate_Putamen', 'Cortex_Category'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Caudate_Putamen', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'net_cau_cat', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=net_df,
                             dv='Connectivity',
                             within=['Cortex_Category', 'Caudate_Putamen'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Cortex_Category', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'net_cat_cau', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=net_df,
                             dv='Connectivity',
                             within=['hemisphere', 'Cortex_Category'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'hemisphere', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'net_hemi_cat', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=net_df,
                             dv='Connectivity',
                             within=['Cortex_Category', 'hemisphere'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Cortex_Category', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'net_cat_hemi', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=net_df,
                             dv='Connectivity',
                             within=['Striatum_ROI', 'hemisphere'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Striatum_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'net_striatum_hemi', STATS_OUT_DIR)


#### Rostral–caudal

In [ ]:
aov = AnovaRM(net_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Rostral_Caudal', 'hemisphere', 'Cortex_Category']).fit()
aov.summary()
export_anova(aov, 'net_rostral_hemi_cat', STATS_OUT_DIR)


##### 2-way: Category x Rostral/Caudal

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = net_df.groupby(['participant_id', 'Cortex_Category', 'Rostral_Caudal'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Cortex_Category', 'Rostral_Caudal'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'net_cat_rostral', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=net_df,
                             dv='Connectivity',
                             within=['hemisphere', 'Rostral_Caudal'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'hemisphere', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'net_hemi_rostral', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=net_df,
                             dv='Connectivity',
                             within=['Rostral_Caudal', 'hemisphere'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Rostral_Caudal', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'net_rostral_hemi', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=net_df,
                             dv='Connectivity',
                             within=['Cortex_Category', 'Rostral_Caudal'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Cortex_Category', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'net_cat_rostral', STATS_OUT_DIR)


#### Direction check: Category x Caudate/Putamen x Rostral/Caudal

In [ ]:
net_means = (net_df.groupby(['Cortex_Category', 'Caudate_Putamen', 'Rostral_Caudal'])['Connectivity']
             .agg(['mean', 'median']))
display(net_means)

for cat in NETWORKS:
    sub = net_means.loc[cat, 'mean']
    top = sub.idxmax()
    print(f'{cat:10s} peaks in {top[1]} {top[0]}  (mean={sub.max():.3g})')

#### Group-mean plot: networks side by side

In [ ]:
# One value per participant x network x striatal subregion, averaged over the
# network's ROIs and both hemispheres (same collapsing as the per-network plots).
df_collapsed_net = net_df.groupby(['participant_id', 'Cortex_Category', 'Striatum_ROI'],
                                   as_index=False).agg({'Connectivity': 'mean'})

fig, ax = plt.subplots(1, 1, figsize=(6, 4), dpi=500)

sns.barplot(data=df_collapsed_net,
            x='Cortex_Category', y='Connectivity',
            order=NETWORKS,
            hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
            palette='crest',
            alpha=0.8, gap=0.1, fill=False,
            ax=ax)
sns.stripplot(data=df_collapsed_net,
              x='Cortex_Category', y='Connectivity',
              order=NETWORKS,
              hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
              palette='crest',
              dodge=True, jitter=True,
              linewidth=0.2, alpha=0.2,
              legend=None,
              ax=ax)

sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1))
ax.set_title(f'Corticostriatal connectivity by network (mean of {num_participants} HCP 7T participants)')
ax.set_xlabel('')
plt.yscale('log')
ax.set_ylabel('connectivity (a.u. / mm³)')

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'streamline-count_x-network_hue-striatum.svg'))

In [ ]:
# Same plot split by hemisphere -- the hemisphere posthocs say THAT the
# hemispheres differ; this shows WHICH side is stronger in each network.
df_collapsed_net_hemi = net_df.groupby(['participant_id', 'hemisphere', 'Cortex_Category', 'Striatum_ROI'],
                                        as_index=False).agg({'Connectivity': 'mean'})

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=500, sharey=True)

for ax, hemi in zip(axes, ['Left', 'Right']):
    d = df_collapsed_net_hemi[df_collapsed_net_hemi['hemisphere'] == hemi]
    sns.barplot(data=d,
                x='Cortex_Category', y='Connectivity',
                order=NETWORKS,
                hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
                palette='crest',
                alpha=0.8, gap=0.1, fill=False,
                legend='auto' if hemi == 'Right' else False,
                ax=ax)
    sns.stripplot(data=d,
                  x='Cortex_Category', y='Connectivity',
                  order=NETWORKS,
                  hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
                  palette='crest',
                  dodge=True, jitter=True,
                  linewidth=0.2, alpha=0.2,
                  legend=None,
                  ax=ax)
    ax.set_title(f'{hemi} hemisphere')
    ax.set_xlabel('')
    ax.set_yscale('log')
    ax.set_ylabel('connectivity (a.u. / mm³)')

sns.move_legend(axes[1], 'upper left', bbox_to_anchor=(1, 1))
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'streamline-count_x-network_hue-striatum_by-hemi.svg'))

In [ ]:
pfc_df = net_df[net_df['Cortex_Category'] == 'Prefrontal']

df_collapsed_pfc = pfc_df.groupby(['participant_id', 'Cortex_ROI', 'Striatum_ROI'],
                                   as_index=False).agg({'Connectivity': 'mean'})

fig, ax = plt.subplots(1, 1, figsize=(8, 4), dpi=500)

sns.barplot(data=df_collapsed_pfc,
            x='Cortex_ROI', y='Connectivity',
            order=PREFRONTAL_ROIS,
            hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
            palette='crest',
            alpha=0.8, gap=0.1, fill=False,
            ax=ax)
sns.stripplot(data=df_collapsed_pfc,
              x='Cortex_ROI', y='Connectivity',
              order=PREFRONTAL_ROIS,
              hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
              palette='crest',
              dodge=True, jitter=True,
              linewidth=0.2, alpha=0.2,
              legend=None,
              ax=ax)

ax.set_xticklabels([_display_label(r) for r in PREFRONTAL_ROIS])
sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1))
ax.set_title(f'Prefrontal–striatal probabilistic connectivity (mean of {num_participants} HCP 7T participants)')
plt.yscale('log')
ax.set_ylabel('connectivity (a.u. / mm³)')

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'streamline-count_x-prefrontal_hue-striatum.svg'))

### PFC statistics

In [ ]:
pfc_long_df = all_long_df[all_long_df['Cortex_ROI'].isin(PREFRONTAL_ROIS)]


#### All factors

In [ ]:
aov = AnovaRM(pfc_long_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI', 'hemisphere']).fit()
aov.summary()
export_anova(aov, 'pfc_full', STATS_OUT_DIR)


#### No hemisphere

In [ ]:
aov = AnovaRM(pfc_long_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI']).fit()
aov.summary()
export_anova(aov, 'pfc_cau_rostral_roi', STATS_OUT_DIR)


#### Caudate–Putamen

In [ ]:
aov = AnovaRM(pfc_long_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Caudate_Putamen', 'Cortex_ROI', 'hemisphere']).fit()
aov.summary()
export_anova(aov, 'pfc_cau_roi_hemi', STATS_OUT_DIR)


##### 2-way: Cortex ROI x Caudate/Putamen

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = pfc_long_df.groupby(['participant_id', 'Cortex_ROI', 'Caudate_Putamen'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Cortex_ROI', 'Caudate_Putamen'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'pfc_roi_cau', STATS_OUT_DIR)


In [ ]:
# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=pfc_long_df,
                             dv='Connectivity',
                             within=['Caudate_Putamen', 'Cortex_ROI'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)

print('Available columns:', pairwise.columns)
print('\nPost-hoc Pairwise Comparisons:')
pd.set_option('display.max_rows', None)

pairwise[['Contrast', 'Caudate_Putamen', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_df = pairwise.copy()
for col in ('A', 'B', 'Cortex_ROI'):
    if col in export_df.columns:
        export_df[col] = export_df[col].replace(ROI_DISPLAY_NAMES)
export_posthoc(export_df, 'pfc_cau_roi', STATS_OUT_DIR)


In [ ]:
# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=pfc_long_df,
                             dv='Connectivity',
                             within=['Cortex_ROI', 'Caudate_Putamen'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)

print('Available columns:', pairwise.columns)
print('\nPost-hoc Pairwise Comparisons:')
pd.set_option('display.max_rows', None)

pairwise[['Contrast', 'Cortex_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_df = pairwise.copy()
for col in ('A', 'B', 'Cortex_ROI'):
    if col in export_df.columns:
        export_df[col] = export_df[col].replace(ROI_DISPLAY_NAMES)
export_posthoc(export_df, 'pfc_roi_cau', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=pfc_long_df,
                             dv='Connectivity',
                             within=['hemisphere', 'Cortex_ROI'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'hemisphere', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'pfc_hemi_roi', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=pfc_long_df,
                             dv='Connectivity',
                             within=['Cortex_ROI', 'hemisphere'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Cortex_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'pfc_roi_hemi', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=pfc_long_df,
                             dv='Connectivity',
                             within=['Striatum_ROI', 'hemisphere'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Striatum_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'pfc_striatum_hemi', STATS_OUT_DIR)


#### Rostral–caudal

In [ ]:
aov = AnovaRM(pfc_long_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Rostral_Caudal', 'hemisphere', 'Cortex_ROI']).fit()
aov.summary()
export_anova(aov, 'pfc_rostral_hemi_roi', STATS_OUT_DIR)


##### 2-way: Cortex ROI x Rostral/Caudal

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = pfc_long_df.groupby(['participant_id', 'Cortex_ROI', 'Rostral_Caudal'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Cortex_ROI', 'Rostral_Caudal'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'pfc_roi_rostral', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=pfc_long_df,
                             dv='Connectivity',
                             within=['hemisphere', 'Rostral_Caudal'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'hemisphere', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'pfc_hemi_rostral', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=pfc_long_df,
                             dv='Connectivity',
                             within=['Rostral_Caudal', 'hemisphere'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Rostral_Caudal', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'pfc_rostral_hemi', STATS_OUT_DIR)


In [ ]:
# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=pfc_long_df,
                             dv='Connectivity',
                             within=['Cortex_ROI', 'Rostral_Caudal'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)

print('Available columns:', pairwise.columns)
print('\nPost-hoc Pairwise Comparisons:')
pd.set_option('display.max_rows', None)

pairwise[['Contrast', 'Cortex_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_df = pairwise.copy()
for col in ('A', 'B', 'Cortex_ROI'):
    if col in export_df.columns:
        export_df[col] = export_df[col].replace(ROI_DISPLAY_NAMES)
export_posthoc(export_df, 'pfc_roi_rostral', STATS_OUT_DIR)


### Visual statistics

In [ ]:
vis_long_df = all_long_df[all_long_df['Cortex_ROI'].isin(VISUAL_ROIS)]

#### Group-mean plot

In [ ]:
df_collapsed_vis = vis_long_df.groupby(['participant_id', 'Cortex_ROI', 'Striatum_ROI'],
                                        as_index=False).agg({'Connectivity': 'mean'})

fig, ax = plt.subplots(1, 1, figsize=(8, 4), dpi=500)

sns.barplot(data=df_collapsed_vis,
            x='Cortex_ROI', y='Connectivity',
            order=VISUAL_ROIS,
            hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
            palette='crest',
            alpha=0.8, gap=0.1, fill=False,
            ax=ax)
sns.stripplot(data=df_collapsed_vis,
              x='Cortex_ROI', y='Connectivity',
              order=VISUAL_ROIS,
              hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
              palette='crest',
              dodge=True, jitter=True,
              linewidth=0.2, alpha=0.2,
              legend=None,
              ax=ax)

ax.set_xticklabels([_display_label(r) for r in VISUAL_ROIS])
sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1))
ax.set_title(f'Visual–striatal probabilistic connectivity (mean of {num_participants} HCP 7T participants)')
plt.yscale('log')
ax.set_ylabel('connectivity (a.u. / mm³)')

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'streamline-count_x-visual_hue-striatum.svg'))

#### All factors

In [ ]:
aov = AnovaRM(vis_long_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI', 'hemisphere']).fit()
aov.summary()
export_anova(aov, 'vis_full', STATS_OUT_DIR)


#### No hemisphere

In [ ]:
aov = AnovaRM(vis_long_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI']).fit()
aov.summary()
export_anova(aov, 'vis_cau_rostral_roi', STATS_OUT_DIR)


#### Caudate–Putamen

In [ ]:
aov = AnovaRM(vis_long_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Caudate_Putamen', 'Cortex_ROI', 'hemisphere']).fit()
aov.summary()
export_anova(aov, 'vis_cau_roi_hemi', STATS_OUT_DIR)


##### 2-way: Cortex ROI x Caudate/Putamen

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = vis_long_df.groupby(['participant_id', 'Cortex_ROI', 'Caudate_Putamen'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Cortex_ROI', 'Caudate_Putamen'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'vis_roi_cau', STATS_OUT_DIR)


In [ ]:
# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=vis_long_df,
                             dv='Connectivity',
                             within=['Caudate_Putamen', 'Cortex_ROI'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)

print('Available columns:', pairwise.columns)
print('\nPost-hoc Pairwise Comparisons:')
pd.set_option('display.max_rows', None)

pairwise[['Contrast', 'Caudate_Putamen', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_df = pairwise.copy()
for col in ('A', 'B', 'Cortex_ROI'):
    if col in export_df.columns:
        export_df[col] = export_df[col].replace(ROI_DISPLAY_NAMES)
export_posthoc(export_df, 'vis_cau_roi', STATS_OUT_DIR)


In [ ]:
# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=vis_long_df,
                             dv='Connectivity',
                             within=['Cortex_ROI', 'Caudate_Putamen'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)

print('Available columns:', pairwise.columns)
print('\nPost-hoc Pairwise Comparisons:')
pd.set_option('display.max_rows', None)

pairwise[['Contrast', 'Cortex_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_df = pairwise.copy()
for col in ('A', 'B', 'Cortex_ROI'):
    if col in export_df.columns:
        export_df[col] = export_df[col].replace(ROI_DISPLAY_NAMES)
export_posthoc(export_df, 'vis_roi_cau', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=vis_long_df,
                             dv='Connectivity',
                             within=['hemisphere', 'Cortex_ROI'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'hemisphere', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'vis_hemi_roi', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=vis_long_df,
                             dv='Connectivity',
                             within=['Cortex_ROI', 'hemisphere'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Cortex_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'vis_roi_hemi', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=vis_long_df,
                             dv='Connectivity',
                             within=['Striatum_ROI', 'hemisphere'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Striatum_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'vis_striatum_hemi', STATS_OUT_DIR)


#### Rostral–caudal

In [ ]:
aov = AnovaRM(vis_long_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Rostral_Caudal', 'hemisphere', 'Cortex_ROI']).fit()
aov.summary()
export_anova(aov, 'vis_rostral_hemi_roi', STATS_OUT_DIR)


##### 2-way: Cortex ROI x Rostral/Caudal

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = vis_long_df.groupby(['participant_id', 'Cortex_ROI', 'Rostral_Caudal'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Cortex_ROI', 'Rostral_Caudal'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'vis_roi_rostral', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=vis_long_df,
                             dv='Connectivity',
                             within=['hemisphere', 'Rostral_Caudal'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'hemisphere', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'vis_hemi_rostral', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=vis_long_df,
                             dv='Connectivity',
                             within=['Rostral_Caudal', 'hemisphere'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Rostral_Caudal', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'vis_rostral_hemi', STATS_OUT_DIR)


In [ ]:
# Pairwise comparisons for region with FDR correction
pairwise = pg.pairwise_tests(data=vis_long_df,
                             dv='Connectivity',
                             within=['Cortex_ROI', 'Rostral_Caudal'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)

print('Available columns:', pairwise.columns)
print('\nPost-hoc Pairwise Comparisons:')
pd.set_option('display.max_rows', None)

pairwise[['Contrast', 'Cortex_ROI', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_df = pairwise.copy()
for col in ('A', 'B', 'Cortex_ROI'):
    if col in export_df.columns:
        export_df[col] = export_df[col].replace(ROI_DISPLAY_NAMES)
export_posthoc(export_df, 'vis_roi_rostral', STATS_OUT_DIR)


### Motor statistics

Positive control: motor cortex -> caudal putamen is one of the best-established corticostriatal pathways. The current atlas has a single motor ROI (PreCG), so there is no ROI factor; the models below add one automatically once `MOTOR_ROIS` has more than one ROI.

In [ ]:
mot_long_df = all_long_df[all_long_df['Cortex_ROI'].isin(MOTOR_ROIS)]
MOT_HAS_ROI = len(MOTOR_ROIS) > 1

#### Group-mean plot

In [ ]:
df_collapsed_mot = mot_long_df.groupby(['participant_id', 'Cortex_ROI', 'Striatum_ROI'],
                                        as_index=False).agg({'Connectivity': 'mean'})

fig, ax = plt.subplots(1, 1, figsize=(max(3, 1.5 * len(MOTOR_ROIS) + 1.5), 4), dpi=500)

sns.barplot(data=df_collapsed_mot,
            x='Cortex_ROI', y='Connectivity',
            order=MOTOR_ROIS,
            hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
            palette='crest',
            alpha=0.8, gap=0.1, fill=False,
            ax=ax)
sns.stripplot(data=df_collapsed_mot,
              x='Cortex_ROI', y='Connectivity',
              order=MOTOR_ROIS,
              hue='Striatum_ROI', hue_order=['aCAU', 'pCAU', 'aPUT', 'pPUT'],
              palette='crest',
              dodge=True, jitter=True,
              linewidth=0.2, alpha=0.2,
              legend=None,
              ax=ax)

sns.move_legend(ax, 'upper left', bbox_to_anchor=(1, 1))
ax.set_title(f'Motor–striatal connectivity\n(mean of {num_participants} HCP 7T participants)')
plt.yscale('log')
ax.set_ylabel('connectivity (a.u. / mm³)')

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'streamline-count_x-motor_hue-striatum.svg'))

#### All factors

In [ ]:
_mot_within = ['Caudate_Putamen', 'Rostral_Caudal', 'hemisphere'] + (['Cortex_ROI'] if MOT_HAS_ROI else [])
aov = AnovaRM(mot_long_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=_mot_within).fit()
aov.summary()
export_anova(aov, 'motor_full', STATS_OUT_DIR)

#### 2-way follow-ups

Striatal division x striatal axis (the motor network's topography), plus ROI x division and ROI x axis when there is more than one motor ROI.

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2), averaging over the
# other factors so there is one value per subject x cell.
_mot_two_way = [(['Caudate_Putamen', 'Rostral_Caudal'], 'motor_cau_rostral')]
if MOT_HAS_ROI:
    _mot_two_way += [(['Cortex_ROI', 'Caudate_Putamen'], 'motor_roi_cau'),
                     (['Cortex_ROI', 'Rostral_Caudal'], 'motor_roi_rostral')]

for _within, _lbl in _mot_two_way:
    _d = mot_long_df.groupby(['participant_id'] + _within,
                             as_index=False)['Connectivity'].mean()
    aov = pg.rm_anova(data=_d, dv='Connectivity', within=_within,
                      subject='participant_id', detailed=True, effsize='np2')
    print(f'\n=== {_lbl} ===')
    display(export_anova(aov, _lbl, STATS_OUT_DIR))

#### Pairwise comparisons

In [ ]:
_mot_pairs = [(['Caudate_Putamen', 'Rostral_Caudal'], 'motor_cau_rostral'),
              (['Rostral_Caudal', 'Caudate_Putamen'], 'motor_rostral_cau'),
              (['Striatum_ROI', 'hemisphere'],        'motor_striatum_hemi'),
              (['hemisphere', 'Rostral_Caudal'],      'motor_hemi_rostral'),
              (['Rostral_Caudal', 'hemisphere'],      'motor_rostral_hemi')]
if MOT_HAS_ROI:
    _mot_pairs += [(['Cortex_ROI', 'Caudate_Putamen'], 'motor_roi_cau'),
                   (['Cortex_ROI', 'Rostral_Caudal'],  'motor_roi_rostral'),
                   (['Cortex_ROI', 'hemisphere'],      'motor_roi_hemi')]

for _within, _lbl in _mot_pairs:
    pairwise = pg.pairwise_tests(data=mot_long_df,
                                 dv='Connectivity',
                                 within=_within,
                                 subject='participant_id',
                                 padjust='fdr', parametric=False)
    print(f'\n=== {_lbl} ===')
    display(export_posthoc(pairwise, _lbl, STATS_OUT_DIR))

In [ ]:
mot_means = (mot_long_df.groupby(['Caudate_Putamen', 'Rostral_Caudal'])['Connectivity']
             .agg(['mean', 'median']))
display(mot_means)
top = mot_means['mean'].idxmax()
print(f'Motor peaks in {top[1]} {top[0]}  (mean={mot_means["mean"].max():.3g})')

## Sensory processing hierarchy (primary vs. secondary)

Hypothesis: primary sensory cortex shows relatively more posterior + putamen
striatal connectivity than secondary/higher-order sensory cortex, within each
modality. ROIs are ranked into ordered hierarchy levels rather than split into
a single primary/secondary binary, for more statistical power and a more
falsifiable trend claim.

- **Auditory** (core-belt-parabelt, Kaas & Hackett 2000): HG = Primary (core);
  PP, PT = Belt; STGa, STGp = Parabelt.
- **Visual** (V1 in the calcarine sulcus, outward through the hierarchy):
  IntraCalc, SupraCalc, OccPole = Primary (all three tile the calcarine
  sulcus/occipital pole in the Harvard-Oxford atlas -- none alone is a
  complete V1 proxy the way Heschl's Gyrus is for A1, and Supracalcarine
  is not meaningfully more "downstream" than Intracalcarine, so they're
  grouped rather than artificially split); LOCsup, LOCinf = Extrastriate
  (lateral occipital); OccFus, TempOccFus = Ventral (fusiform, higher-order
  ventral stream). Note: Harvard-Oxford's Cuneal Cortex and Lingual Gyrus
  also border the calcarine sulcus and are natural "Primary" candidates but
  aren't in this atlas's 46 cortical ROIs -- a gap worth addressing in a
  future atlas rebuild, not fixable from this notebook alone.

**v2 atlas** (`ATLAS_VERSION = 'v2'`): cuneal cortex and lingual gyrus join the visual Primary level; temporo-occipital MTG and ITG (MT+ region) form a fourth, lateral occipitotemporal level (LatOccTemp); posterior MTG and temporal pole form a fourth auditory level (Association).

These orderings are anatomically motivated, not a validated distance metric --
adjust the dicts below if you'd place any ROI differently. Auditory and
visual are tested separately (different anatomical bases per modality; not
combined into one factor) -- convergence between the two is itself evidence
for the hypothesis.

In [ ]:
AUD_HIERARCHY = {'HG': 'Primary', 'PP': 'Belt', 'PT': 'Belt',
                 'STGa': 'Parabelt', 'STGp': 'Parabelt'}
VIS_HIERARCHY = {'IntraCalc': 'Primary', 'SupraCalc': 'Primary', 'OccPole': 'Primary',
                 'LOCsup': 'Extrastriate', 'LOCinf': 'Extrastriate',
                 'OccFus': 'Ventral', 'TempOccFus': 'Ventral'}
if ATLAS_VERSION == 'v2':
    # posterior MTG and temporal pole: auditory association cortex beyond parabelt
    AUD_HIERARCHY.update({'MTGp': 'Association', 'TempPole': 'Association'})
    # cuneus and lingual gyrus tile the calcarine banks with IntraCalc/SupraCalc;
    # temporo-occipital MTG/ITG (MT+ region) form a lateral occipitotemporal level
    VIS_HIERARCHY.update({'Cuneal': 'Primary', 'Lingual': 'Primary',
                          'MTGto': 'LatOccTemp', 'ITGto': 'LatOccTemp'})

### Auditory hierarchy

In [ ]:
aud_hier_df = aud_long_df.copy()
aud_hier_df['Hierarchy'] = aud_hier_df['Cortex_ROI'].map(AUD_HIERARCHY)


#### Hierarchy x Caudate/Putamen

In [ ]:
aov = AnovaRM(aud_hier_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Hierarchy', 'Caudate_Putamen', 'Rostral_Caudal']).fit()
aov.summary()
export_anova(aov, 'aud_hier_cau_rostral', STATS_OUT_DIR)


In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = aud_hier_df.groupby(['participant_id', 'Hierarchy', 'Caudate_Putamen'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Hierarchy', 'Caudate_Putamen'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'aud_hier_cau', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=aud_hier_df,
                             dv='Connectivity',
                             within=['Caudate_Putamen', 'Hierarchy'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Caudate_Putamen', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'aud_cau_hier', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=aud_hier_df,
                             dv='Connectivity',
                             within=['Hierarchy', 'Caudate_Putamen'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Hierarchy', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'aud_hier_cau', STATS_OUT_DIR)


#### Hierarchy x Rostral/Caudal

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = aud_hier_df.groupby(['participant_id', 'Hierarchy', 'Rostral_Caudal'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Hierarchy', 'Rostral_Caudal'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'aud_hier_rostral', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=aud_hier_df,
                             dv='Connectivity',
                             within=['Hierarchy', 'Rostral_Caudal'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Hierarchy', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'aud_hier_rostral', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=aud_hier_df,
                             dv='Connectivity',
                             within=['Rostral_Caudal', 'Hierarchy'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Rostral_Caudal', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'aud_rostral_hier', STATS_OUT_DIR)


### Visual hierarchy

In [ ]:
vis_hier_df = vis_long_df.copy()
vis_hier_df['Hierarchy'] = vis_hier_df['Cortex_ROI'].map(VIS_HIERARCHY)


#### Hierarchy x Caudate/Putamen

In [ ]:
aov = AnovaRM(vis_hier_df,
              aggregate_func='mean',
              depvar='Connectivity',
              subject='participant_id',
              within=['Hierarchy', 'Caudate_Putamen', 'Rostral_Caudal']).fit()
aov.summary()
export_anova(aov, 'vis_hier_cau_rostral', STATS_OUT_DIR)


In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = vis_hier_df.groupby(['participant_id', 'Hierarchy', 'Caudate_Putamen'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Hierarchy', 'Caudate_Putamen'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'vis_hier_cau', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=vis_hier_df,
                             dv='Connectivity',
                             within=['Caudate_Putamen', 'Hierarchy'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Caudate_Putamen', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'vis_cau_hier', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=vis_hier_df,
                             dv='Connectivity',
                             within=['Hierarchy', 'Caudate_Putamen'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Hierarchy', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'vis_hier_cau', STATS_OUT_DIR)


#### Hierarchy x Rostral/Caudal

In [ ]:
# 2-way RM-ANOVA (Greenhouse-Geisser + partial eta^2): the omnibus that the
# pairwise comparisons below follow up. Average over the other factors so
# there is one value per subject x cell.
_d = vis_hier_df.groupby(['participant_id', 'Hierarchy', 'Rostral_Caudal'],
                     as_index=False)['Connectivity'].mean()
aov = pg.rm_anova(data=_d, dv='Connectivity',
                  within=['Hierarchy', 'Rostral_Caudal'],
                  subject='participant_id', detailed=True, effsize='np2')
display(aov)
export_anova(aov, 'vis_hier_rostral', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=vis_hier_df,
                             dv='Connectivity',
                             within=['Hierarchy', 'Rostral_Caudal'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Hierarchy', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'vis_hier_rostral', STATS_OUT_DIR)


In [ ]:
pairwise = pg.pairwise_tests(data=vis_hier_df,
                             dv='Connectivity',
                             within=['Rostral_Caudal', 'Hierarchy'],
                             subject='participant_id',
                             padjust='fdr', parametric=False)
pairwise[['Contrast', 'Rostral_Caudal', 'A', 'B', 'W_val', 'p_unc', 'p_corr', 'hedges']]
export_posthoc(pairwise, 'vis_rostral_hier', STATS_OUT_DIR)


#### Direction check: Hierarchy x Caudate/Putamen x Rostral/Caudal

In [ ]:
for prefix, df in [('Auditory', aud_hier_df), ('Visual', vis_hier_df)]:
    print(f'--- {prefix} ---')
    means = (df.groupby(['Hierarchy', 'Caudate_Putamen', 'Rostral_Caudal'])['Connectivity']
             .agg(['mean', 'median']))
    display(means)
    for level in df['Hierarchy'].unique():
        sub = means.loc[level, 'mean']
        top = sub.idxmax()
        print(f'  {level:14s} peaks in {top[1]} {top[0]}  (mean={sub.max():.3g})')


### Hierarchy on proportional connectivity

The tests above compare absolute connectivity, which mixes two things: how strongly an ROI connects to striatum overall, and where in striatum that connectivity goes. The hypothesis is about the second. Here each participant x ROI x hemisphere's connectivity is normalised to its total across the four striatal subregions, and the hierarchy levels are compared on the share that reaches caudal putamen (pPUT).

In [ ]:
def _pput_share(df):
    """Share of each participant x ROI x hemisphere's striatal connectivity
    that reaches pPUT, averaged over the ROIs within each hierarchy level."""
    keys = ['participant_id', 'Hierarchy', 'Cortex_ROI', 'hemisphere']
    wide = df.pivot_table(index=keys, columns='Striatum_ROI',
                          values='Connectivity', aggfunc='mean')
    total = wide.sum(axis=1)
    share = (wide['pPUT'] / total.where(total > 0)).rename('pPUT_share').reset_index()
    n_drop = int(share['pPUT_share'].isna().sum())
    if n_drop:
        print(f'  dropping {n_drop} participant x ROI x hemisphere cells with zero total connectivity')
    share = share.dropna(subset=['pPUT_share'])
    return share.groupby(['participant_id', 'Hierarchy'], as_index=False)['pPUT_share'].mean()

for prefix, df in [('aud', aud_hier_df), ('vis', vis_hier_df)]:
    print(f'--- {prefix} ---')
    prop = _pput_share(df)
    display(prop.groupby('Hierarchy')['pPUT_share'].agg(['mean', 'median', 'std']))

    aov = pg.rm_anova(data=prop, dv='pPUT_share', within='Hierarchy',
                      subject='participant_id', detailed=True, effsize='np2')
    display(export_anova(aov, f'{prefix}_hier_prop', STATS_OUT_DIR))

    pairwise = pg.pairwise_tests(data=prop, dv='pPUT_share', within='Hierarchy',
                                 subject='participant_id',
                                 padjust='fdr', parametric=False)
    display(export_posthoc(pairwise, f'{prefix}_hier_prop', STATS_OUT_DIR))
    export_cell_means(prop, 'Hierarchy', f'{prefix}_hier_prop', STATS_OUT_DIR, dv='pPUT_share')

## Cell means: direction of effects

The ANOVA and Wilcoxon exports say whether conditions differ, not which is larger (Wilcoxon W is the smaller of the two signed-rank sums). These tables record per-cell means and medians of the participant-level values that enter the tests, so each effect's direction -- including every hemisphere asymmetry -- is documented. Saved as `means_<label>.tsv`.

In [ ]:
_cell_mean_specs = []
for _prefix, _df in [('aud', aud_long_df), ('pfc', pfc_long_df),
                     ('vis', vis_long_df), ('motor', mot_long_df)]:
    _cell_mean_specs += [
        (_df, ['hemisphere'],                       f'{_prefix}_hemi'),
        (_df, ['Caudate_Putamen'],                  f'{_prefix}_cau'),
        (_df, ['Rostral_Caudal'],                   f'{_prefix}_rostral'),
        (_df, ['Striatum_ROI', 'hemisphere'],       f'{_prefix}_striatum_hemi'),
        (_df, ['Cortex_ROI', 'Striatum_ROI'],       f'{_prefix}_roi_striatum'),
        (_df, ['Cortex_ROI', 'hemisphere'],         f'{_prefix}_roi_hemi'),
        (_df, ['Cortex_ROI', 'Striatum_ROI', 'hemisphere'], f'{_prefix}_roi_striatum_hemi'),
    ]
_cell_mean_specs += [
    (net_df, ['Cortex_Category'],                                  'net_cat'),
    (net_df, ['Cortex_Category', 'hemisphere'],                    'net_cat_hemi'),
    (net_df, ['Cortex_Category', 'Caudate_Putamen'],               'net_cat_cau'),
    (net_df, ['Cortex_Category', 'Rostral_Caudal'],                'net_cat_rostral'),
    (net_df, ['Cortex_Category', 'Striatum_ROI'],                  'net_cat_striatum'),
    (net_df, ['Cortex_Category', 'Striatum_ROI', 'hemisphere'],    'net_cat_striatum_hemi'),
    (net_df, ['Rostral_Caudal', 'hemisphere'],                     'net_rostral_hemi'),
    (aud_hier_df, ['Hierarchy', 'Striatum_ROI'],                   'aud_hier_striatum'),
    (vis_hier_df, ['Hierarchy', 'Striatum_ROI'],                   'vis_hier_striatum'),
]
for _df, _by, _lbl in _cell_mean_specs:
    export_cell_means(_df, _by, _lbl, STATS_OUT_DIR)
print(f'wrote {len(_cell_mean_specs)} means_*.tsv tables to {STATS_OUT_DIR}')

# Hemisphere asymmetry at a glance (sign of Left - Right per network)
display(export_cell_means(net_df, ['Cortex_Category', 'hemisphere'], 'net_cat_hemi', STATS_OUT_DIR)
        .pivot(index='Cortex_Category', columns='hemisphere', values=['mean', 'median']))

## Sensitivity analysis — log-transformed connectivity

Self-contained repeat of the omnibus ANOVAs (every network, the network comparison, and the hierarchy models) on `log(C + c)`, with
`c = ½ · min(non-zero C)` so exact-zero edges map to a finite floor. Builds its
own `*_log_df` frames and touches no existing variable. Outputs get a `_log`
suffix and sit alongside the untransformed `anova_*.tsv`.

Whether to adopt this over the untransformed analysis depends on (a) the
exact-zero fraction (see the *Plot distributions* section) and (b) the residual
normality check below.

In [ ]:
# log-transformed copies of the analysis frames (nothing existing is mutated)
_pos = all_long_df.loc[all_long_df['Connectivity'] > 0, 'Connectivity']
LOG_OFFSET = 0.5 * float(_pos.min())
_zero_frac = float((all_long_df['Connectivity'] == 0).mean())
print(f'log offset c = {LOG_OFFSET:.4g}   ({_zero_frac:.1%} of edges are exactly zero)')

def _to_log(df):
    d = df.dropna(subset=['Connectivity']).copy()
    d['Connectivity'] = np.log(d['Connectivity'].clip(lower=0) + LOG_OFFSET)
    return d

all_log_df = _to_log(all_long_df)
aud_log_df = all_log_df[all_log_df['Cortex_ROI'].isin(AUDITORY_ROIS)]
pfc_log_df = all_log_df[all_log_df['Cortex_ROI'].isin(PREFRONTAL_ROIS)]
vis_log_df = all_log_df[all_log_df['Cortex_ROI'].isin(VISUAL_ROIS)]
mot_log_df = all_log_df[all_log_df['Cortex_ROI'].isin(MOTOR_ROIS)]
net_log_df = _to_log(net_df)
aud_hier_log_df = _to_log(aud_hier_df)
vis_hier_log_df = _to_log(vis_hier_df)

In [ ]:
# RM-ANOVA assumes normal residuals after removing subject and condition means.
import scipy.stats as sps
_factors = ['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI', 'hemisphere']

def _rm_resid(df):
    y = df['Connectivity'].to_numpy(float)
    y = y - df.groupby('participant_id')['Connectivity'].transform('mean').to_numpy()
    cell = df[_factors].astype(str).agg('-'.join, axis=1).to_numpy()
    return y - pd.Series(y).groupby(cell).transform('mean').to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (frame, tag) in zip(axes,
                            [(all_long_df.dropna(subset=['Connectivity']), 'raw C'),
                             (all_log_df, 'log(C + c)')]):
    r = _rm_resid(frame)
    r = r[np.isfinite(r)]
    sps.probplot(r, plot=ax)
    _r = r if len(r) <= 5000 else np.random.default_rng(0).choice(r, 5000, replace=False)
    W, p = sps.shapiro(_r)
    ax.set_title(f'{tag}   residuals: Shapiro W = {W:.3f}, p = {p:.1e}')
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'residual_qq_raw_vs_log.svg'))


In [ ]:
# omnibus ANOVAs on the log scale.  >2 within-factors -> AnovaRM; 2 -> rm_anova.
def _omnibus_log(df, within, label):
    if len(within) > 2:
        aov = AnovaRM(df, aggregate_func='mean', depvar='Connectivity',
                      subject='participant_id', within=within).fit()
        return export_anova(aov, label, STATS_OUT_DIR)
    d = df.groupby(['participant_id'] + within, as_index=False)['Connectivity'].mean()
    aov = pg.rm_anova(data=d, dv='Connectivity', within=within,
                      subject='participant_id', detailed=True, effsize='np2')
    return export_anova(aov, label, STATS_OUT_DIR)

_specs = [
    (aud_log_df, ['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI', 'hemisphere'], 'aud_full_log'),
    (aud_log_df, ['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI'],               'aud_cau_rostral_roi_log'),
    (aud_log_df, ['Cortex_ROI', 'Caudate_Putamen'],                                 'aud_roi_cau_log'),
    (aud_log_df, ['Cortex_ROI', 'Rostral_Caudal'],                                  'aud_roi_rostral_log'),
    (pfc_log_df, ['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI', 'hemisphere'], 'pfc_full_log'),
    (pfc_log_df, ['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI'],               'pfc_cau_rostral_roi_log'),
    (pfc_log_df, ['Cortex_ROI', 'Caudate_Putamen'],                                 'pfc_roi_cau_log'),
    (pfc_log_df, ['Cortex_ROI', 'Rostral_Caudal'],                                  'pfc_roi_rostral_log'),
    (vis_log_df, ['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI', 'hemisphere'], 'vis_full_log'),
    (vis_log_df, ['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_ROI'],               'vis_cau_rostral_roi_log'),
    (vis_log_df, ['Cortex_ROI', 'Caudate_Putamen'],                                 'vis_roi_cau_log'),
    (vis_log_df, ['Cortex_ROI', 'Rostral_Caudal'],                                  'vis_roi_rostral_log'),
    (mot_log_df, _mot_within,                                                       'motor_full_log'),
    (mot_log_df, ['Caudate_Putamen', 'Rostral_Caudal'],                             'motor_cau_rostral_log'),
    (net_log_df, ['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_Category', 'hemisphere'], 'net_full_log'),
    (net_log_df, ['Caudate_Putamen', 'Rostral_Caudal', 'Cortex_Category'],          'net_cau_rostral_cat_log'),
    (net_log_df, ['Cortex_Category', 'Caudate_Putamen'],                            'net_cat_cau_log'),
    (net_log_df, ['Cortex_Category', 'Rostral_Caudal'],                             'net_cat_rostral_log'),
    (aud_hier_log_df, ['Hierarchy', 'Caudate_Putamen', 'Rostral_Caudal'],           'aud_hier_cau_rostral_log'),
    (aud_hier_log_df, ['Hierarchy', 'Caudate_Putamen'],                             'aud_hier_cau_log'),
    (aud_hier_log_df, ['Hierarchy', 'Rostral_Caudal'],                              'aud_hier_rostral_log'),
    (vis_hier_log_df, ['Hierarchy', 'Caudate_Putamen', 'Rostral_Caudal'],           'vis_hier_cau_rostral_log'),
    (vis_hier_log_df, ['Hierarchy', 'Caudate_Putamen'],                             'vis_hier_cau_log'),
    (vis_hier_log_df, ['Hierarchy', 'Rostral_Caudal'],                              'vis_hier_rostral_log'),
]
for _df, _within, _lbl in _specs:
    print(f'\n=== {_lbl} ===')
    display(_omnibus_log(_df, _within, _lbl))

## Anatomical Connectivity Figure (Yeterian & Pandya 1998 Style)

Schematic figure showing auditory corticostriatal connectivity on a lateral brain view and coronal striatal cross-sections.


In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import PathPatch, FancyArrowPatch
from matplotlib.path import Path
import matplotlib.patches as mpatches
import nibabel as nib
from skimage import measure
import seaborn as sns

In [ ]:
# Per-ROI mask files (from atlas_construction.ipynb) + an MNI brain mask.
atlas_dir = os.path.join('/Users/dsj3886/data_local/derivatives/', ATLAS_BASE)
file_prefix = ATLAS_BASE

BRAIN_MASK_PATH = os.path.join('/Users/dsj3886/data_local/reference/',
                               'mni_icbm152_nlin_asym_09c_nifti/mni_icbm152_nlin_asym_09c/',
                               'mni_icbm152_t1_tal_nlin_asym_09c_mask.nii')

AUDITORY_MASK_PATHS = {
    "HG":   os.path.join(atlas_dir, f'{file_prefix}_L-HG.nii.gz'),
    "PP":   os.path.join(atlas_dir, f'{file_prefix}_L-PP.nii.gz'),
    "PT":   os.path.join(atlas_dir, f'{file_prefix}_L-PT.nii.gz'),
    "STGa": os.path.join(atlas_dir, f'{file_prefix}_L-STGa.nii.gz'),
    "STGp": os.path.join(atlas_dir, f'{file_prefix}_L-STGp.nii.gz'),
}

PREFRONTAL_MASK_PATHS = {
    "OFC":   os.path.join(atlas_dir, f'{file_prefix}_L-OFC.nii.gz'),
    "ACC":   os.path.join(atlas_dir, f'{file_prefix}_L-ACC.nii.gz'),
    "FMC":   os.path.join(atlas_dir, f'{file_prefix}_L-FMC.nii.gz'),
    "MFG":   os.path.join(atlas_dir, f'{file_prefix}_L-MFG.nii.gz'),
    "IFGt":  os.path.join(atlas_dir, f'{file_prefix}_L-IFGt.nii.gz'),
    "IFGo":  os.path.join(atlas_dir, f'{file_prefix}_L-IFGo.nii.gz'),
    "PreCG": os.path.join(atlas_dir, f'{file_prefix}_L-PreCG.nii.gz'),
}

STRIATAL_MASK_PATHS = {
    "aPUT": os.path.join(atlas_dir, f'{file_prefix}_aPUT-lh.nii.gz'),
    "pPUT": os.path.join(atlas_dir, f'{file_prefix}_pPUT-lh.nii.gz'),
    "aCAU": os.path.join(atlas_dir, f'{file_prefix}_aCAU-lh.nii.gz'),
    "pCAU": os.path.join(atlas_dir, f'{file_prefix}_pCAU-lh.nii.gz'),
}

In [ ]:
def get_roi_contour(mask_path, axis=1, slice_idx=None):
    """
    Load a binary NIfTI mask and return the largest 2D contour from a slice.

    axis=0 -> sagittal slice
    axis=1 -> coronal slice
    slice_idx: if None, use the slice with the largest cross-sectional area.
    """
    data = nib.load(mask_path).get_fdata()
    if slice_idx is None:
        areas = [np.take(data, i, axis=axis).sum() for i in range(data.shape[axis])]
        slice_idx = int(np.argmax(areas))
    slice_2d = np.take(data, slice_idx, axis=axis)
    contours = measure.find_contours(slice_2d, level=0.5)
    return max(contours, key=len)


def get_best_slice_idx(mask_path, axis):
    """Return the slice index with the largest mask area along the given axis."""
    data = nib.load(mask_path).get_fdata()
    areas = [np.take(data, i, axis=axis).sum() for i in range(data.shape[axis])]
    return int(np.argmax(areas))


In [ ]:

CORTEX_ROI      = 'HG'   # highlighted in the brain panel

all_conn = (
    all_long_df[all_long_df['Cortex_ROI'].isin(CORTEX_ROIS)]
    .groupby(['Cortex_ROI', 'Striatum_ROI'])['Connectivity']
    .mean()
    .reset_index()
    .rename(columns={'Connectivity': 'conn_mean'})
)
print(all_conn)


In [ ]:
from matplotlib.colors import Normalize as MplNorm, LogNorm, to_rgb
from matplotlib.cm import ScalarMappable

CMAP = plt.cm.YlOrRd

AUDITORY_PALETTE = {
    'HG':   CMAP(0.85),
    'PP':   '#5aafe0',
    'PT':   '#78c679',
    'STGa': '#9970ab',
    'STGp': '#d9a63a',
}

PREFRONTAL_PALETTE = {
    'OFC':   '#e8a838',
    'ACC':   '#d46b6b',
    'FMC':   '#8e5fbf',
    'MFG':   '#4d9de0',
    'IFGt':  '#3bb273',
    'IFGo':  '#e84855',
    'PreCG': '#7f8c8d',
}

# Paint order: last = highest priority (drawn on top).
# Auditory (temporal plane) are given priority over prefrontal.
PAINT_ORDER = ['PreCG', 'IFGo', 'IFGt', 'MFG', 'FMC', 'ACC', 'OFC',
               'STGp', 'STGa', 'PT', 'PP', 'HG']

# Z-shear applied before Y-Z projection.
# Positive value: each voxel at X=x is shifted +x*SHEAR_FACTOR in Z.
# This tilts the projection to look from slightly below, separating the
# superior temporal plane (HG/PP/PT, deeper/more medial) from the lateral
# surface (STGa/STGp, more lateral).
SHEAR_FACTOR = 0.3


def _sheared_footprint(mask_3d, shear_factor, out_nz):
    """Y-Z footprint with Z-shear. Returns bool array (ny, out_nz)."""
    nx, ny, nz = mask_3d.shape
    result = np.zeros((ny, out_nz), dtype=bool)
    for x in range(nx):
        z_shift = int(round(x * shear_factor))
        sl = mask_3d[x] > 0          # (ny, nz)
        src_end = min(nz, out_nz - z_shift)
        dst_start = z_shift
        if src_end > 0 and dst_start < out_nz:
            result[:, dst_start:dst_start + src_end] |= sl[:, :src_end]
    return result


def _make_surface_rgba(brain_mask_path, roi_mask_paths, palette):
    """
    Sheared Y-Z footprint per ROI painted in PAINT_ORDER.
    Returns (rgba (nz_out,ny,4), brain_proj (ny,nz_out), roi_label (ny,nz_out), roi_list).
    """
    brain_data = nib.load(brain_mask_path).get_fdata() > 0
    nx, ny, nz = brain_data.shape
    nz_out = nz + int(nx * SHEAR_FACTOR) + 2   # extra Z room after shearing

    brain_proj = _sheared_footprint(brain_data, SHEAR_FACTOR, nz_out)

    footprints = {
        roi: _sheared_footprint(nib.load(path).get_fdata(), SHEAR_FACTOR, nz_out)
        for roi, path in roi_mask_paths.items()
    }

    rgba = np.zeros((nz_out, ny, 4), dtype=float)
    in_brain = brain_proj.T
    rgba[in_brain, :3] = 0.85
    rgba[in_brain,  3] = 1.0

    roi_list  = list(roi_mask_paths.keys())
    roi_label = np.zeros((ny, nz_out), dtype=int)

    for roi in PAINT_ORDER:
        if roi not in footprints:
            continue
        fp   = footprints[roi]
        mask = fp.T
        rgba[mask, :3] = to_rgb(palette.get(roi, '#888888'))
        rgba[mask,  3] = 1.0
        roi_label[fp] = roi_list.index(roi) + 1

    rgba[~in_brain, 3] = 0.0
    return rgba, brain_proj, roi_label, roi_list, footprints


def draw_brain_panel(ax, brain_mask_path, cortex_mask_paths, striatal_mask_paths):
    combined_palette = {**AUDITORY_PALETTE, **PREFRONTAL_PALETTE}
    rgba, brain_proj, roi_label, roi_list, footprints = _make_surface_rgba(
        brain_mask_path, cortex_mask_paths, combined_palette)

    ny, nz_out = brain_proj.shape
    ax.imshow(rgba, origin='lower', extent=[0, ny, 0, nz_out],
              aspect='equal', interpolation='nearest')

    # Brain silhouette
    brain_contour = max(measure.find_contours(brain_proj.astype(float), level=0.5), key=len)
    ax.plot(brain_contour[:, 0], brain_contour[:, 1], color='black', linewidth=1.5, zorder=4)

    # ROI boundary contours
    for roi, fp in footprints.items():
        contours = measure.find_contours(fp.astype(float), level=0.5)
        for c in contours:
            ax.plot(c[:, 0], c[:, 1], color='black', linewidth=0.6, zorder=5)

    # Labels at painted centroid
    for idx, roi in enumerate(roi_list, start=1):
        ys, zs = np.where(roi_label == idx)
        if len(ys) == 0:
            continue
        is_active = (roi == CORTEX_ROI)
        ax.text(ys.mean(), zs.mean(), _display_label(roi),
                ha='center', va='center', fontsize=7, zorder=6,
                fontweight='bold' if is_active else 'normal',
                color='white' if is_active else 'black')

    # Crop to brain bounding box
    ys, zs = np.where(brain_proj)
    margin = 4
    ax.set_xlim(ys.min() - margin, ys.max() + margin)
    ax.set_ylim(zs.min() - margin, zs.max() + margin)
    ax.set_aspect('equal')
    ax.invert_xaxis()

    ax.axis('off')


def draw_striatal_structure(ax, roi_ant, roi_post, hg_conn_df, striatal_mask_paths, norm, title=None):
    data_ant  = nib.load(striatal_mask_paths[roi_ant]).get_fdata()
    data_post = nib.load(striatal_mask_paths[roi_post]).get_fdata()
    slice_idx = int(np.argmax((data_ant + data_post).sum(axis=(1, 2))))

    for roi, data in [(roi_ant, data_ant), (roi_post, data_post)]:
        row      = hg_conn_df[hg_conn_df["Striatum_ROI"] == roi]
        conn_val = float(row["conn_mean"].values[0]) if not row.empty else 0
        contours = measure.find_contours(data[slice_idx, :, :], level=0.5)
        if not contours:
            continue
        contour = max(contours, key=len)
        verts = np.column_stack([contour[:, 0], contour[:, 1]])
        codes  = [Path.MOVETO] + [Path.LINETO] * (len(verts) - 2) + [Path.CLOSEPOLY]
        ax.add_patch(PathPatch(Path(verts, codes),
                               facecolor=CMAP(norm(conn_val)),
                               edgecolor='black', linewidth=1.5))
        label = 'ant' if roi == roi_ant else 'post'
        ax.text(np.mean(contour[:, 0]), np.mean(contour[:, 1]), label,
                ha='center', va='center', fontsize=6, color='black', alpha=0.7)

    if title:
        ax.set_title(title, fontsize=10)
    ax.set_aspect('equal')
    ax.autoscale()
    # Tighten limits to the data so no internal whitespace inflates the gap
    margin = 3
    ax.set_xlim(ax.get_xlim()[0] - margin, ax.get_xlim()[1] + margin)
    ax.set_ylim(ax.get_ylim()[0] - margin, ax.get_ylim()[1] + margin)
    ax.invert_xaxis()
    ax.axis('off')


In [ ]:
# -- Build figure: one striatal section per network --------------------------
from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable

vmax     = float(all_conn['conn_mean'].max())
vmin_log = float(all_conn.loc[all_conn['conn_mean'] > 0, 'conn_mean'].min())
norm     = LogNorm(vmin=vmin_log, vmax=vmax)

SECTIONS = [('Auditory',   AUDITORY_ROIS),
            ('Prefrontal', PREFRONTAL_ROIS),
            ('Visual',     VISUAL_ROIS),
            ('Motor',      MOTOR_ROIS)]

col_w = 1.8
n_cols = max(len(rois) for _, rois in SECTIONS)
fig_w = col_w * n_cols
fig   = plt.figure(figsize=(fig_w, 3.8 * len(SECTIONS)))

# Outer grid: one row per network, each holding a caudate row + putamen row.
# Every section uses the same number of columns so a striatal panel is the
# same size in every row; shorter networks leave the right-hand columns empty.
gs_outer = gridspec.GridSpec(len(SECTIONS), 1, height_ratios=[1.8] * len(SECTIONS), hspace=0.3)

section_axes = []   # (name, first caudate ax, first putamen ax, caudate axes)
for row, (name, rois) in enumerate(SECTIONS):
    gs_sec = gridspec.GridSpecFromSubplotSpec(
        2, n_cols, subplot_spec=gs_outer[row], wspace=0.06, hspace=0.15)
    cau_axes, put_axes = [], []
    for col, roi in enumerate(rois):
        conn_df = all_conn[all_conn['Cortex_ROI'] == roi]
        ax_cau = fig.add_subplot(gs_sec[0, col])
        ax_put = fig.add_subplot(gs_sec[1, col])
        draw_striatal_structure(ax_cau, 'aCAU', 'pCAU', conn_df, STRIATAL_MASK_PATHS, norm)
        draw_striatal_structure(ax_put, 'aPUT', 'pPUT', conn_df, STRIATAL_MASK_PATHS, norm)
        ax_cau.set_title(_display_label(roi), fontsize=9)
        cau_axes.append(ax_cau)
        put_axes.append(ax_put)

    cau_axes[0].annotate('Caudate', xy=(0, 0.5), xycoords='axes fraction',
                         xytext=(-8, 0), textcoords='offset points',
                         ha='right', va='center', fontsize=9, rotation=90)
    put_axes[0].annotate('Putamen', xy=(0, 0.5), xycoords='axes fraction',
                         xytext=(-8, 0), textcoords='offset points',
                         ha='right', va='center', fontsize=9, rotation=90)
    section_axes.append((name, cau_axes[0], put_axes[0], cau_axes))

# ── Section labels ────────────────────────────────────────────────────────────
fig.canvas.draw()
# Each section label sits above its row of column titles -- measure the
# titles' actual rendered top (they vary with fontsize/dpi) rather than
# guessing a fixed fig-fraction pad, which previously undershot and let the
# label overlap whichever ROI title landed in the middle column.
renderer = fig.canvas.get_renderer()
for name, _, _, cau_axes in section_axes:
    title_top = max(
        ax.title.get_window_extent(renderer).transformed(fig.transFigure.inverted()).y1
        for ax in cau_axes
    )
    fig.text(0.5, title_top + 0.005, name, ha='center', va='bottom', fontsize=10, style='italic')

# ── Shared colorbar ──────────────────────────────────────────────────────────
sm = ScalarMappable(cmap=CMAP, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.93, 0.2, 0.012, 0.25])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('connectivity (a.u. / mm³)', fontsize=8)

fig.suptitle('Corticostriatal connectivity', fontsize=12, y=1.01)

fig.savefig(os.path.join(FIGURES_DIR, 'corticostriatal_connectivity_schematic.pdf'), bbox_inches='tight', dpi=300)
fig.savefig(os.path.join(FIGURES_DIR, 'corticostriatal_connectivity_schematic.png'), bbox_inches='tight', dpi=300)
fig.savefig(os.path.join(FIGURES_DIR, 'corticostriatal_connectivity_schematic.svg'), bbox_inches='tight', dpi=300)
plt.show()

### Surface ROI visualisation (Destrieux approximation)

Renders the auditory / prefrontal ROIs on an fsaverage surface for figure panels. Uses approximate Destrieux labels, independent of the connectivity analysis.

In [ ]:
from nilearn import datasets, plotting, surface
import numpy as np

destrieux = datasets.fetch_atlas_surf_destrieux()
fsaverage = datasets.fetch_surf_fsaverage('fsaverage5')

labels = [l.decode() if isinstance(l, bytes) else l for l in destrieux['labels']]
parc   = destrieux['map_left']
coords, _ = surface.load_surf_mesh(fsaverage['infl_left'])

# ROI → Destrieux label
auditory_destrieux = {
    'STG':  'G_temp_sup-Lateral',
    'HG':   'G_temp_sup-G_T_transv',
    'PP':   'G_temp_sup-Plan_polar',
    'PT':   'G_temp_sup-Plan_tempo',
}
lateral_pfc_destrieux = {
    'PreCG': 'G_precentral',
    'MFG':   'G_front_middle',
    'IFGt':  'G_front_inf-Triangul',
    'IFGo':  'G_front_inf-Opercular',
}
medial_pfc_destrieux = {
    'FMC': 'G_and_S_frontomargin',
    'ACC': 'G_and_S_cingul-Ant',
    'OFC': 'G_orbital',
}


def make_roi_map(roi_dict):
    roi_map = np.zeros(len(parc))
    for roi_id, label in enumerate(roi_dict.values(), start=1):
        idx = labels.index(label)
        roi_map[parc == idx] = roi_id
    return roi_map


def add_labels(ax, roi_map, roi_dict):
    for roi_id, name in enumerate(roi_dict.keys(), start=1):
        verts = np.where(roi_map == roi_id)[0]
        if len(verts) == 0:
            continue
        cx, cy, cz = coords[verts].mean(axis=0)
        ax.text(cx, cy, cz, _display_label(name), fontsize=9, fontweight='bold',
                ha='center', va='center', color='black',
                bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.))


# ── Lateral view: auditory + lateral prefrontal ───────────────────────────────
lateral_rois    = {**auditory_destrieux, **lateral_pfc_destrieux}
roi_map_lateral = make_roi_map(lateral_rois)

fig_lat = plotting.plot_surf_roi(
    fsaverage['infl_left'],
    roi_map=roi_map_lateral,
    bg_map=fsaverage['sulc_left'],
    bg_on_data=True,
    darkness=0.5,
    view='lateral',
    title='Lateral — auditory & lateral prefrontal',
)
add_labels(fig_lat.axes[0], roi_map_lateral, lateral_rois)

# ── Medial view: medial/orbital prefrontal ────────────────────────────────────
roi_map_medial = make_roi_map(medial_pfc_destrieux)

fig_med = plotting.plot_surf_roi(
    fsaverage['infl_left'],
    roi_map=roi_map_medial,
    bg_map=fsaverage['sulc_left'],
    bg_on_data=True,
    darkness=0.5,
    view='medial',
    title='Medial — prefrontal (FMC, ACC, OFC)',
)
add_labels(fig_med.axes[0], roi_map_medial, medial_pfc_destrieux)


In [ ]:
import os
atlas_fpath = os.path.join('/Users/dsj3886/data_local/derivatives/', ATLAS_BASE,
                           f'{ATLAS_BASE}_atlas_space-MNI152NLin6Asym.nii.gz')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from nilearn.plotting import view_img_on_surf
import nibabel as nib

# Keep the striatal subregions and every cortical ROI that enters the
# analysis (CORTEX_ROIS); zero out cortical labels outside it and the whole
# right hemisphere. Selecting by name rather than by label index keeps this
# correct when the atlas gains ROIs.
atlas_img  = nib.load(atlas_fpath)
atlas_data = atlas_img.get_fdata().copy()
for label_idx, label_name in enumerate(region_list, start=1):
    is_cortical = label_name[:2] in ('L-', 'R-')
    if label_name.startswith('R-') or (is_cortical and label_name[2:] not in CORTEX_ROIS):
        atlas_data[atlas_data == label_idx] = 0

# Remap surviving labels to a dense 1..k range -- raw label indices (up to 37)
# leave ~9 unused gaps from zeroed-out regions, so the hue wheel below was
# being split into more slices than there are visible regions, packing real
# colors closer together than necessary.
present = sorted(int(v) for v in np.unique(atlas_data) if v > 0)
remap = {old: new for new, old in enumerate(present, start=1)}
remapped = np.zeros_like(atlas_data)
for old_idx, new_idx in remap.items():
    remapped[atlas_data == old_idx] = new_idx
atlas_data = remapped

masked_img = nib.Nifti1Image(atlas_data.astype(np.float32), atlas_img.affine, atlas_img.header)
n_labels = len(present)

# Evenly-spaced hues (HUSL) at fixed saturation/lightness -- guarantees a
# minimum hue separation between all n_labels colors (no near-duplicate
# "browns") and stays off white/black by construction (unlike randomly
# sampling from tab20/tab20b/tab20c, which include pale and near-duplicate
# earth-tone entries). husl assigns hues in index order, so labels that are
# close together in the LUT (e.g. HG-PP-PT-STGa-STGp = 9-13, or ACC/OFC =
# 35/37) get near-adjacent hues and read as one blob even though they're
# spatially adjacent, distinct regions -- shuffle the hue assignment (fixed
# seed) so index-neighbors don't get color-neighbors.
# Hue-shuffling alone can't raise the *minimum* pairwise hue distance -- with
# n_labels evenly-spaced hues, some pair is always exactly one hue-step apart
# no matter how they're assigned (all slots are used). Alternate lightness by
# hue-slot position so whichever two labels land on adjacent hues (unavoidable)
# still differ visibly in brightness, independent of shuffle luck.
_hues_dark  = sns.husl_palette(n_labels, l=0.45)
_hues_light = sns.husl_palette(n_labels, l=0.72)
_hues = [(_hues_dark if i % 2 == 0 else _hues_light)[i] for i in range(n_labels)]
_perm = np.random.default_rng(0).permutation(n_labels)
cmap_rois = ListedColormap([_hues[i] for i in _perm])

view = view_img_on_surf(
    stat_map_img=masked_img,
    surf_mesh="fsaverage",
    threshold=0.5,
    cmap=cmap_rois,
    vmin=1,
    vmax=n_labels,
    vol_to_surf_kwargs={
        "n_samples": 1,
        "radius": 3,
        "interpolation": "nearest",
    },
    symmetric_cmap=False,
    colorbar=False,
)

#view  # interactive; save manually, then clear this output (keeps the notebook small)

In [ ]:
# Static, savable re-render of the interactive view above, sampling the SAME
# volumetric atlas. Uses inner_mesh (white matter surface) rather than a
# radius-ball, so each vertex's sample stays within the local cortical ribbon
# instead of jumping across sulcal folds into an unrelated gyrus (visible
# earlier as speckled/interdigitated regions, worst medially). n_samples=1
# keeps it a single nearest-voxel lookup per vertex -- averaging across depth
# samples would be meaningless for discrete integer labels.
texture = surface.vol_to_surf(
    masked_img, fsaverage['pial_left'],
    inner_mesh=fsaverage['white_left'],
    interpolation='nearest',
    n_samples=1,
)

_views = ['lateral', 'medial', 'dorsal', 'ventral', 'anterior', 'posterior']
for _view in _views:
    _fig = plotting.plot_surf_roi(
        fsaverage['infl_left'], roi_map=texture, hemi='left', view=_view,
        bg_map=fsaverage['sulc_left'], bg_on_data=True, darkness=0.5,
        cmap=cmap_rois, vmin=1, vmax=n_labels,
        title=f'Atlas ROIs -- {_view}',
    )
    _fig.savefig(os.path.join(FIGURES_DIR, f'atlas_surface_{_view}.png'), dpi=300, bbox_inches='tight')
    plt.close(_fig)

print(f'saved {len(_views)} static views to {FIGURES_DIR}/atlas_surface_*.png')


In [ ]:
import numpy as np
vals, counts = np.unique(texture, return_counts=True)
for v, n in zip(vals, counts):
    if v > 0:
        print(f'label {int(v):3d}  {region_list[int(v)-1]:8s}  {n} vertices  color={cmap_rois(int(v)/n_labels)}')
